# Group Report Chapter 5: Tables and Charts
* Supporting tables and charts to be used for the `Chapter 5 Analysis and Results`

In [1]:
# Import statements
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

In [2]:
# File paths (current: src/)
RESULTS = Path('../results')
DATA = Path('../data/NSW')


In [3]:
# Palette set explicitly for figures
SURFACE, INK, INK2, MUTED, GRID, AXIS = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
BLUE, ORANGE, RED, BLUE_LIGHT = '#2a78d6', '#eb6834', '#e34948', '#9ec5f4'
plt.style.use('default')
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'text.color': INK, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.edgecolor': AXIS, 'grid.color': GRID, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

MODELS = ['lightgbm', 'lstm', 'xgboost', 'random_forest', 'prophet', 'baseline']
NAMES = {'lightgbm': 'LightGBM', 'lstm': 'LSTM', 'xgboost': 'XGBoost', 'random_forest': 'Random forest',
         'prophet': 'Prophet', 'baseline': 'Linear regression', 'lightgbm_tuned': 'LightGBM (tuned)',
         'naive_yesterday': 'Naive, same time yesterday'}
SEGMENTS = {'overall': 'All of 2019', 'Summer': 'Summer', 'Autumn': 'Autumn', 'Winter': 'Winter',
            'Spring': 'Spring', 'above_18': '18°C and above', 'below_18': 'Below 18°C',
            'is_school_term': 'School term', 'is_school_holiday': 'School holiday'}
SEASON = {1: 'Summer', 2: 'Summer', 12: 'Summer', 3: 'Autumn', 4: 'Autumn', 5: 'Autumn',
          6: 'Winter', 7: 'Winter', 8: 'Winter', 9: 'Spring', 10: 'Spring', 11: 'Spring'}
fmt = lambda d=1: (lambda v: f'{v:,.{d}f}')

In [4]:
# Get Actual

## Load Results

In [5]:
# Segment metrics saved by model_comparison.ipynb; the main results are the runs with radiation_day_before
seg = pd.read_csv(RESULTS / 'model_comparison_by_segment.csv')
main = seg[seg.with_radiation]
by_segment = lambda metric: main.pivot_table(index='model_name', columns='segment', values=metric).loc[MODELS, list(SEGMENTS)]

# Actual 2019 demand, observed temperature, school flag and AEMO forecasts (unscaled)
raw = pd.read_csv(DATA / 'nsw_features_added.csv', parse_dates=['DATETIME']).set_index('DATETIME').sort_index()
y2019 = raw.loc['2019']
actual = y2019['TOTALDEMAND']

# Saved 2019 predictions, aligned to the same half-hours (Prophet's file is written day-first)
def load_predictions(name):
    f = pd.read_csv(RESULTS / f'{name}_predictions.csv')
    dates = f.iloc[:, 0].astype(str)
    f.index = pd.to_datetime(dates, format='ISO8601') if dates.str.match(r'^\d{4}-').all() else pd.to_datetime(dates, dayfirst=True)
    return f.iloc[:, 1].reindex(actual.index)

preds = pd.DataFrame({m: load_predictions(m) for m in MODELS + ['lightgbm_tuned']})
preds['naive'] = raw['TOTALDEMAND'].shift(48).reindex(actual.index)
preds['aemo_dayprior'] = y2019['forecast_dayprior']
preds['aemo_12hr'] = y2019['forecast_12hr_prior']
print(f'{len(seg)} segment rows | {len(actual):,} half-hours in 2019 | LSTM covers {preds.lstm.notna().sum():,}')

108 segment rows | 17,520 half-hours in 2019 | LSTM covers 17,461


In [6]:
# Supporting: accuracy metrics, and the Diebold-Mariano test (absolute loss, Newey-West over one day,
# Harvey-Leybourne-Newbold correction).
# Note: A negative DM statistic means forecast 'a' is more accurate.
def metrics(pred):
    v = pd.concat([actual, pred], axis=1).dropna(); a, p = v.iloc[:, 0], v.iloc[:, 1]
    return {'MAE (MW)': mean_absolute_error(a, p), 'RMSE (MW)': np.sqrt(mean_squared_error(a, p)),
            'MAPE (%)': mean_absolute_percentage_error(a, p) * 100, 'R2': r2_score(a, p), 'Bias (MW)': (a - p).mean()}

def diebold_mariano(a, b, h=48):
    v = pd.concat([actual, a, b], axis=1).dropna()
    d = (v.iloc[:, 0] - v.iloc[:, 1]).abs() - (v.iloc[:, 0] - v.iloc[:, 2]).abs()
    n = len(d)
    var = d.var() * (1 + 2 * sum((1 - k / h) * d.autocorr(k) for k in range(1, h))) / n
    dm = d.mean() / np.sqrt(var) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return {'Mean |e| difference (MW)': d.mean(), 'DM': dm, 'p-value': 2 * stats.t.sf(abs(dm), df=n - 1)}

# Labels for each 2019 half-hour, matching model_comparison.ipynb
masks = {'overall': np.ones(len(actual), bool),
         **{s: (actual.index.month.map(SEASON) == s) for s in ['Summer', 'Autumn', 'Winter', 'Spring']},
         'above_18': (y2019['TEMPERATURE'] >= 18).values, 'below_18': (y2019['TEMPERATURE'] < 18).values,
         'is_school_term': y2019['is_school_term'].astype(bool).values,
         'is_school_holiday': ~y2019['is_school_term'].astype(bool).values}
temp = raw['TEMPERATURE']
change = pd.cut((temp - temp.shift(48)).abs().reindex(actual.index), [0, 2, 5, 8, np.inf], right=False,
                labels=['Under 2°C', '2 to 5°C', '5 to 8°C', '8°C or more'])

## Tables

In [7]:
# Table 5.1 Model accuracy, with the naive forecast for reference
t51 = pd.DataFrame({NAMES[m]: metrics(preds[m]) for m in MODELS + ['naive']}).T
t51['Skill vs linear regression (%)'] = (1 - t51['MAE (MW)'] / t51.loc['Linear regression', 'MAE (MW)']) * 100
print('Table 5.1 Model accuracy, 2019 test year'); print(t51.to_string(float_format=fmt(2)))

KeyError: 'naive'